## Submission Instructions

You must submit two materials for HW3 on KLMS **by May 30th at 23:59:59**, with penalty for late submission only over two days \(-20%, -40%\).

<br>

### A. ```.ipynb``` file containing your Python code

- Provide your code below each problem. After completing each problem, make sure to **run the code and print the results**. Your submitted code must be **executable**.
- You are allowed to use more than one code block per problem.

### B. Report

- Submit a report that describes the process and results of solving each problem (either with screenshots or text).
- There is no page limit, but the report must be in PDF format.
- Avoid unnecessary explanations for simple problems, but make sure to **include essential details** on how you solved each problem.
- The report must be written **in English only**. Other languages are not allowed.

### C. File Naming Convention
Your file names must follow the format:
- ```CS372_HW3_{studentID}.ipynb```
- ```CS372_HW3_{studentID}.pdf```

<br>

Any form of plagiarism will be penalized.

If you have any questions, please use the KLMS Q&A board or contact cs372@nlp.kaist.ac.kr.

## Environment Setting

The downloads and imports below are baseline settings for the assignment. Once you are connected to the server, you must download these files; otherwise, you may encounter errors such as "Resource words not found."

If you need additional libraries or packages for solving the problems, you are allowed to use them. However, you **must** mention why you used them in your report.

In [1]:
# Execution done

# # Download dataset from Github
# !git clone https://github.com/google-research-datasets/gap-coreference

Cloning into 'gap-coreference'...
remote: Enumerating objects: 19, done.
remote: Total 19 (delta 0), reused 0 (delta 0), pack-reused 19 (from 1)
Receiving objects: 100% (19/19), 970.68 KiB | 14.93 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [1]:
from collections import defaultdict
import csv
from enum import Enum
import pandas as pd

## Problem (48 points)

Write Python code to solve a coreference resolution problem. In this task, the goal is to accurately match an ambiguous pronoun to its correct coreferent name when potential coreference names are provided. For example, consider the sentence:


> *MacKenzie* studied with *Bernard Leach* from 1949 to 1952. **His** simple, wheel-thrown functional pottery is heavily influenced by the oriental aesthetic of Shoji Hamada and Kanjiro Kawai.


In this example, the bolded "**his**" has two candidate coreferents: "MacKenzie" and "Bernard Leach." Since "**his**" refers to "MacKenzie" in this context, the expected output should be TRUE for "MacKenzie" and FALSE for "Bernard Leach." Note that the ambiguous pronoun may refer to both candidate names or to neither of them.

Obtain the results for this problem under two different task settings: snippet-context and page-context. In the snippet-context setting, do not use the context from the given Wikipedia URL. In the page-context setting, use the context from the corresponding Wikipedia page.

For the implementation, you should not use external models, in particular those specialized for coreference resolution, NER, or entity linking.

Your report should include the following:
- The methods used to achieve the goal, along with the intuition and hypotheses underlying these methods
- A performance table for each task setting, including precision, recall, F1 score, and bias ratio
- Additional proposed methods to further improve the results

### Step 0: Evaluation Code (**DO NOT MODIFY**)

In [2]:
# Do not modify this part

class Gender(Enum):
    UNKNOWN = 0
    MASCULINE = 1
    FEMININE = 2


# Mapping of (lowercased) pronoun form to gender value. Note that reflexives are not included in GAP, so do not appear here.
PRONOUNS = {
    'she': Gender.FEMININE,
    'her': Gender.FEMININE,
    'hers': Gender.FEMININE,
    'he': Gender.MASCULINE,
    'his': Gender.MASCULINE,
    'him': Gender.MASCULINE,
}

# Fieldnames used in the gold dataset .tsv file.
GOLD_FIELDNAMES = [
    'ID', 'Text', 'Pronoun', 'Pronoun-offset', 'A', 'A-offset', 'A-coref', 'B',
    'B-offset', 'B-coref', 'URL'
]

# Fieldnames expected in system output .tsv files.
SYSTEM_FIELDNAMES = ['ID', 'A-coref', 'B-coref']

class Annotation(object):
    """
    Container class for storing annotations of an example.
    Attributes:
        gender (None): The gender of the annotation. None indicates that gender was not determined for the given example.
        name_a_coref (None): bool reflecting whether Name A was recorded as coreferential with the target pronoun for this example. None indicates that no annotation was found for the given example.
        name_b_coref (None): bool reflecting whether Name B was recorded as coreferential with the target pronoun for this example. None indicates that no annotation was found for the given example.
    """

    def __init__(self):
        self.gender = None
        self.name_a_coref = None
        self.name_b_coref = None


class Scores(object):
    """
    Container class for storing scores, and generating evaluation metrics.
    Attributes:
        true_positives: Tally of true positives seen.
        false_positives: Tally of false positives seen.
        true_negatives: Tally of true negatives seen.
        false_negatives: Tally of false negatives seen.
    """

    def __init__(self):
        self.true_positives = 0
        self.false_positives = 0
        self.true_negatives = 0
        self.false_negatives = 0

    def recall(self):
        """
        Calculate recall based on the observed scores.

        Return:
        recall (float)
        """
        numerator = self.true_positives
        denominator = self.true_positives + self.false_negatives
        return 100.0 * numerator / denominator if denominator else 0.0

    def precision(self):
        """
        Calculate precision based on the observed scores.

        Return:
        precision (float)
        """
        numerator = self.true_positives
        denominator = self.true_positives + self.false_positives
        return 100.0 * numerator / denominator if denominator else 0.0

    def f1(self):
        """
        Calculate F1 score based on the observed scores.

        Return:
        F1 score (float)
        """
        recall = self.recall()
        precision = self.precision()

        numerator = 2 * precision * recall
        denominator = precision + recall
        return numerator / denominator if denominator else 0.0

In [3]:
def read_annotations(filename, is_gold):
    """
    Read coreference annotations for the examples in the given file.

    Args:
    filename: Path to .tsv file to read.
    is_gold: Whether or not we are reading the gold annotations.

    Return:
    A dict mapping example ID strings to their Annotation representation. If reading gold, 'Pronoun' field is used to determine gender.
    """

    def is_true(value):
        if value.lower() == 'true':
            return True
        elif value.lower() == 'false':
            return False
        else:
            print('Unexpected label!', value)
            return None

    fieldnames = GOLD_FIELDNAMES if is_gold else SYSTEM_FIELDNAMES

    annotations = defaultdict(Annotation)
    with open(filename, newline='') as f:
        reader = csv.DictReader(f, fieldnames=fieldnames, delimiter='\t')

        # Skip the header line in the gold data
        if is_gold:
            next(reader, None)

        for row in reader:
            example_id = row['ID']
            if example_id in annotations:
                print('Multiple annotations for', example_id)
                continue

            annotations[example_id].name_a_coref = is_true(row['A-coref'])
            annotations[example_id].name_b_coref = is_true(row['B-coref'])
            if is_gold:
                gender = PRONOUNS.get(row['Pronoun'].lower(), Gender.UNKNOWN)
                assert gender != Gender.UNKNOWN, row
                annotations[example_id].gender = gender

    return annotations


def calculate_scores(gold_annotations, system_annotations):
    """
    Score the system annotations against gold.

    Args:
    gold_annotations: dict from example ID to its gold Annotation.
    system_annotations: dict from example ID to its system Annotation.

    Return:
    A dict from gender to a Scores object for that gender. None is used to denote no specific gender, i.e. overall scores.
    """
    scores = {}
    for example_id, gold_annotation in gold_annotations.items():
        system_annotation = system_annotations[example_id]

        name_a_annotations = [
            gold_annotation.name_a_coref, system_annotation.name_a_coref
        ]
        name_b_annotations = [
            gold_annotation.name_b_coref, system_annotation.name_b_coref
        ]
        for gender in [None, gold_annotation.gender]:
            if gender not in scores:
                scores[gender] = Scores()

            for (gold, system) in [name_a_annotations, name_b_annotations]:
                if system is None:
                    print('Missing output for', example_id)
                    scores[gender].false_negatives += 1
                elif gold and system:
                    scores[gender].true_positives += 1
                elif not gold and system:
                    scores[gender].false_positives += 1
                elif not gold and not system:
                    scores[gender].true_negatives += 1
                elif gold and not system:
                    scores[gender].false_negatives += 1

    return scores


def make_score_card(scores):
    """
    Return a human-readable score_card of the given scores.

    Args:
    scores: dict from gender to its Scores object. None is used to denote no specific gender, i.e. overall scores.

    Return:
    score_card (string)
    """
    score_card = []

    display_names = [(None, 'Overall'), (Gender.MASCULINE, 'Masculine'),
                    (Gender.FEMININE, 'Feminine')]

    bias_terms = {}
    for gender, display_name in display_names:
        gender_scores = scores.get(gender, Scores())

        recall = gender_scores.recall()
        precision = gender_scores.precision()
        f1 = gender_scores.f1()
        bias_terms[gender] = f1

        score_card.append('{} recall: {:.1f} precision: {:.1f} f1: {:.1f}'.format(
            display_name, recall, precision, f1))
        score_card.append('\t\ttp {:d}\tfp {:d}'.format(
            gender_scores.true_positives, gender_scores.false_positives))
        score_card.append('\t\tfn {:d}\ttn {:d}'.format(
            gender_scores.false_negatives, gender_scores.true_negatives))

    bias = '-'
    if bias_terms[Gender.MASCULINE] and bias_terms[Gender.FEMININE]:
        bias = '{:.2f}'.format(
            bias_terms[Gender.FEMININE] / bias_terms[Gender.MASCULINE])

    score_card.append('Bias (F/M): {}\n'.format(bias))
    return '\n'.join(score_card)


def run_scorer(gold_tsv, system_tsv):
    """
    Run the scorer.

    Args:
    gold_tsv: Gold annotations to score against.
    system_tsv: System output to score.

    Return:
    score_card (string)
    """
    gold_annotations = read_annotations(gold_tsv, is_gold=True)
    assert gold_annotations, 'No gold annotations read!'

    system_annotations = read_annotations(system_tsv, is_gold=False)
    assert system_annotations, 'No system annotations read!'

    scores = calculate_scores(gold_annotations, system_annotations)
    return make_score_card(scores)

### Step 1: Data Acquisition & File Path Setup

In [4]:
import traceback

try:
    data = pd.read_csv('gap-coreference/gap-test.tsv', sep='\t')
except Exception as e:
    print(">>> FULL TRACEBACK:")
    traceback.print_exc()

In [5]:
gold_tsv = '/gap-coreference/gap-test.tsv' # file path for test data
predict_tsv = '/output/result.tsv' # file path for your prediction results

### Step 2: Ambiguity Resolution

Write Python code that resolves ambiguity in pronouns referring to names.

In [ ]:
# Example

def get_distance(pronoun_offset, name_offset):
    return abs(pronoun_offset - name_offset)

def resolve_coref_snippet(row):
    pronoun_offset = row['Pronoun-offset']
    a_offset = row['A-offset']
    b_offset = row['B-offset']

    distance_a = get_distance(pronoun_offset, a_offset)
    distance_b = get_distance(pronoun_offset, b_offset)

    a_coref = distance_a < distance_b
    b_coref = distance_b < distance_a

    if distance_a == distance_b:
        a_coref = True
        b_coref = False

    return a_coref, b_coref

### Step 3: Result Saving

Save the prediction results to the prediction tsv file path defined above. The tsv file should contain the test ID and a True or False value for A-coref and B-coref.

In [ ]:
# Example

id = data['ID']
A_coref = []
B_coref = []
for i in range(2000):
    A_coref.append('True')
    B_coref.append('False')

dict = {'ID' : id, 'A_coref' : A_coref, 'B_coref' : B_coref}
df = pd.DataFrame(dict)

with open('result.tsv', 'w', encoding='utf-8', newline='') as f:
    tw = csv.writer(f, delimiter='\t')
    for i in range(2000):
        tw.writerow([id[i], A_coref[i], B_coref[i]])

### Step 4: Evaluation

Check your results here. Feel free to modify this part if necessary.

In [ ]:
score_card = run_scorer(gold_tsv, predict_tsv)
print(score_card)